In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 固定亂數，方便重現
np.random.seed(42)
tf.random.set_seed(42)

In [6]:
def build_target_only_model(input_shape=(100, 6), num_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])
    
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [7]:
def run_target_only_baseline(X_data, y_data, stroke_name, metric_name, test_size=0.2):
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data,
        test_size=test_size,
        random_state=42,
        stratify=y_data
    )
    
    model = build_target_only_model(input_shape=X_data.shape[1:])
    
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )
    
    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=100,
        batch_size=8,
        callbacks=[early_stop],
        verbose=0
    )
    
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n=== {stroke_name} / {metric_name} / target-only ===")
    print("train size:", len(X_train))
    print("test size:", len(X_test))
    print("accuracy:", round(acc, 4))
    print(classification_report(y_test, y_pred, digits=4))
    
    return {
        "球種": stroke_name,
        "指標": metric_name,
        "accuracy": acc,
        "train_size": len(X_train),
        "test_size": len(X_test)
    }, model, history

In [8]:
挑_idx = merged_df["stroke_type_x"] == "挑"
推_idx = merged_df["stroke_type_x"] == "推"

X_挑 = X[挑_idx]
X_推 = X[推_idx]

y_list = [y1, y2, y3, y4, y5]
metric_names = [
    "揮拍軌跡正確度",
    "揮拍速度流暢度",
    "手腕轉動時機正確度",
    "擊球時機正確度",
    "擊球位置正確度"
]

NameError: name 'merged_df' is not defined

In [ ]:
target_only_results = []

for y_all, metric_name in zip(y_list, metric_names):
    y_挑 = y_all[挑_idx]
    y_推 = y_all[推_idx]
    
    result_挑, _, _ = run_target_only_baseline(X_挑, y_挑, "挑", metric_name)
    result_推, _, _ = run_target_only_baseline(X_推, y_推, "推", metric_name)
    
    target_only_results.extend([result_挑, result_推])

target_only_df = pd.DataFrame(target_only_results)
display(target_only_df)

In [ ]:
target_only_pivot = target_only_df.pivot(
    index="指標",
    columns="球種",
    values="accuracy"
).reset_index()

target_only_pivot.columns = ["指標", "挑_target_only_acc", "推_target_only_acc"]
display(target_only_pivot)